In [ ]:
import os
import timm
import torch
import torchvision

import torch.nn              as nn
import torch.optim           as optim
import numpy                 as np
import pandas                as pd
import torchvision.models    as models

from   pathlib               import Path
from   collections           import Counter
from   torch.utils.data      import DataLoader
from   torchvision           import transforms
from   torchvision.io        import ImageReadMode
from   tqdm                  import tqdm
from   utils.process_results import process_metrics

In [ ]:
class BalancedClassDataset(torch.utils.data.Dataset):
    def __init__(self, clean_paths, artifact_paths, transform=None, augment_transform=None, balance=False):
        self.image_paths       = []
        self.labels            = []
        self.transform         = transform
        self.augment_transform = augment_transform
        
        for clean_dir in clean_paths:
            for subdir, _, files in os.walk(clean_dir):
                for file in files:
                    if file.endswith('.png'):
                        file_path = os.path.join(subdir, file)
                        self.image_paths.append(file_path)
                        self.labels.append(0)
        
        for artifact_dir in artifact_paths:
            for subdir, _, files in os.walk(artifact_dir):
                for file in files:
                    if file.endswith('.png'):
                        file_path = os.path.join(subdir, file)
                        self.image_paths.append(file_path)
                        self.labels.append(1)
        
        if balance:
            label_counts    = Counter(self.labels)
            max_count       = max(label_counts.values())
            
            balanced_paths  = []
            balanced_labels = []
            
            for label in [0, 1]:
                label_indices = [i for i, l in enumerate(self.labels) if l == label]
                current_count = len(label_indices)
                
                balanced_paths.extend([self.image_paths[i] for i in label_indices])
                balanced_labels.extend([label] * current_count)
                
                if current_count < max_count:
                    needed              = max_count - current_count
                    oversampled_indices = np.random.choice(label_indices, needed, replace=True)

                    balanced_paths.extend([self.image_paths[i] for i in oversampled_indices])
                    balanced_labels.extend([label] * needed)
            
            self.image_paths = balanced_paths
            self.labels      = balanced_labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image      = torchvision.io.read_image(image_path, mode=ImageReadMode.RGB).float() / 255.0
        label      = self.labels[idx]
        
        if self.augment_transform and np.random.random() > 0.5:
            image = self.augment_transform(image)
        elif self.transform:
            image = self.transform(image)
        
        return image, label

def returnDataLoader(basepath, classes, batch_size, is_training=False):
    basepath = Path(basepath)

    class0   = [basepath / classes[0]]
    class1   = [basepath / c for c in classes[1]] if isinstance(classes[1], list) else [basepath / classes[1]]

    base_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    augment_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    if is_training:
        dataset    = BalancedClassDataset(class0, class1, 
                                       transform=base_transform, 
                                       augment_transform=augment_transform,
                                       balance=True)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    else:
        dataset    = BalancedClassDataset(class0, class1, 
                                       transform=base_transform,
                                       balance=False)
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    return dataloader


In [9]:
class BinaryClassifier(nn.Module):
    def __init__(self, backbone, num_features):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(num_features, 1)
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

In [22]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0
    
    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        
        optimizer.zero_grad()

        outputs = model(images)
        loss    = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        predictions   = (torch.sigmoid(outputs) > 0.5).float()
        correct      += (predictions == labels).sum().item()
        total        += labels.size(0)
    
    return running_loss / len(dataloader), correct / total

In [23]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader):
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            
            outputs = model(images)
            loss    = criterion(outputs, labels)
            
            running_loss += loss.item()
            predictions   = (torch.sigmoid(outputs) > 0.5).float()
            correct      += (predictions == labels).sum().item()
            total        += labels.size(0)
    
    return running_loss / len(dataloader), correct / total

In [ ]:
def evaluate_and_save(model, test_loader, device, output_csv):
    all_true = []
    all_pred = []
    all_prob = []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"Evaluating {output_csv}"):
            images = images.to(device)
            labels = labels.cpu().numpy()
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy().flatten()
            preds = (probs > 0.5).astype(int)
            all_true.extend(labels)
            all_pred.extend(preds)
            all_prob.extend(probs)
    results_df = pd.DataFrame({
        'True_Label': all_true,
        'Predicted_Label': all_pred,
        'Prediction_Probability': all_prob
    })
    results_df.to_csv(output_csv, index=False)
    print(f"Predictions saved to '{output_csv}'.")
    print(results_df)

In [ ]:
data = []

for split in ['training', 'validation', 'test']:
    split_path = Path(f'./data/{split}')
    if split_path.exists():
        for class_folder in split_path.iterdir():
            if class_folder.is_dir():
                count = len(list(class_folder.glob('*.png')))
                data.append({'split': split, 'class': class_folder.name, 'count': count})


In [ ]:
df = pd.DataFrame(data)
print(df.pivot(index='class', columns='split', values='count').fillna(0).astype(int))

In [ ]:
basepath = Path('./data/training')
class0   = [basepath / 'artifact_free']
class1   = [basepath / c for c in ['blood', 'blur', 'bubble', 'damage', 'fold', 'marker']]

dataset  = BalancedClassDataset(class0, class1, balance=True)

print(f"Total samples: {len(dataset)}")
print(f"Class 0 (clean): {dataset.labels.count(0)}")
print(f"Class 1 (artifacts): {dataset.labels.count(1)}")

In [ ]:
uni = timm.create_model( "hf-hub:MahmoodLab/uni", pretrained =True, init_values=1e-5, dynamic_img_size=True )

for name, param in uni.named_parameters(): 
    if "blocks.11" in name or "blocks.12" in name: 
        param.requires_grad = True 
    else: 
        param.requires_grad = False

In [ ]:
uni_classifier = BinaryClassifier(uni, num_features=1024) 

device         = torch.device('mps' if torch.cuda.is_available() else 'cpu')
model          = uni_classifier.to(device)

criterion      = nn.BCEWithLogitsLoss()
optimizer      = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
num_epochs     = 10

In [ ]:
basepath_train = Path('./data/training')
basepath_val   = Path('./data/validation')
basepath_test  = Path('.data/test')

class0         = [basepath_train / 'artifact_free']
class1         = [basepath_train / c for c in ['blood', 'blur', 'bubble', 'damage', 'fold', 'marker']]

train_loader   = returnDataLoader(basepath_train, ['artifact_free', ['blood', 'blur', 'bubble', 'damage', 'fold', 'marker']], batch_size=32, is_training=True)
val_loader     = returnDataLoader(basepath_val, ['artifact_free', ['blood', 'blur', 'bubble', 'damage', 'fold', 'marker']], batch_size=32, is_training=False)
test_loader    = returnDataLoader(basepath_test, ['artifact_free', ['blood', 'blur', 'bubble', 'damage', 'fold', 'marker']], batch_size=32, is_training=False)


In [ ]:
for epoch in tqdm(range(num_epochs), desc="Training"):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc     = validate(model, val_loader, criterion, device)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

In [26]:
torch.save(model.state_dict(), "fma_augmented.pth")

In [41]:
basepath_test = Path('./data/test')
test_loader   = returnDataLoader(basepath_test, ['artifact_free', ['blood', 'blur', 'bubble', 'damage', 'fold', 'marker']], batch_size=32, is_training=False)

In [ ]:
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

In [ ]:
res_model = models.resnet50(pretrained=True)

in_features  = res_model.fc.in_features
res_model.fc = nn.Linear(in_features, 1)

device      = torch.device("mps" if torch.cuda.is_available() else "cpu")
res_model.to(device)

criterion   = nn.BCEWithLogitsLoss()
optimizer   = optim.Adam(model.parameters(), lr=1e-4)
epochs      = 10


/Users/alex/programming/python/HistoART/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/alex/programming/python/HistoART/.venv/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
for epoch in range(epochs):
    print(f'Starting Epoch {epoch + 1}')
    res_model.train()
    
    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in train_loader:
        
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)

        optimizer.zero_grad()

        outputs = res_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        predicted     = (torch.sigmoid(outputs) > 0.5).float()
        correct      += (predicted == labels).sum().item()
        total        += labels.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}')

    res_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
            outputs = res_model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    val_epoch_loss = val_loss / len(val_loader.dataset)
    val_epoch_acc = val_correct / val_total
    print(f'Validation Loss: {val_epoch_loss:.4f}, Validation Accuracy: {val_epoch_acc:.4f}')



In [31]:
torch.save(res_model.state_dict(), "dla_augmented.pth")

In [ ]:
res_model    = models.resnet50(pretrained=False)
in_features  = res_model.fc.in_features
res_model.fc = nn.Linear(in_features, 1)

res_model.load_state_dict(torch.load("dla_augmented.pth", map_location=device))
res_model.to(device)
res_model.eval()

In [ ]:
uni       = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)
fma_model = BinaryClassifier(uni, num_features=1024)

fma_model.load_state_dict(torch.load("fma_augmented.pth", map_location=device))
fma_model.to(device)
fma_model.eval()

In [ ]:
evaluate_and_save(fma_model, test_loader, device, "fma_augmented_results.csv")

In [2]:
metrics = process_metrics("fma_augmented_results.csv")

print("Accuracy:  {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['accuracy'][0],  *metrics['accuracy'][1]))
print("Precision: {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['precision'][0], *metrics['precision'][1]))
print("Recall:    {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['recall'][0],    *metrics['recall'][1]))
print("F1 Score:  {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['f1_score'][0],  *metrics['f1_score'][1]))
print("AUC:       {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['auc_mean'],     *metrics['auc_ci']))

Accuracy:  0.956 (95% CI: 0.951-0.959)
Precision: 0.996 (95% CI: 0.995-0.998)
Recall:    0.954 (95% CI: 0.950-0.958)
F1 Score:  0.975 (95% CI: 0.972-0.977)
AUC:       0.992 (95% CI: 0.990-0.993)


In [ ]:
evaluate_and_save(res_model, test_loader, device, "dla_augmented_results.csv")

In [4]:
metrics = process_metrics("dla_augmented_results.csv")

print("Accuracy:  {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['accuracy'][0],  *metrics['accuracy'][1]))
print("Precision: {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['precision'][0], *metrics['precision'][1]))
print("Recall:    {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['recall'][0],    *metrics['recall'][1]))
print("F1 Score:  {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['f1_score'][0],  *metrics['f1_score'][1]))
print("AUC:       {:.3f} (95% CI: {:.3f}-{:.3f})".format(metrics['auc_mean'],     *metrics['auc_ci']))

Accuracy:  0.861 (95% CI: 0.855-0.867)
Precision: 0.911 (95% CI: 0.906-0.916)
Recall:    0.938 (95% CI: 0.934-0.942)
F1 Score:  0.925 (95% CI: 0.921-0.928)
AUC:       0.637 (95% CI: 0.619-0.653)
